In [2]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, f1_score, root_mean_squared_error

clf_df = pd.read_csv('classification_escalation.csv')
reg_df = pd.read_csv('resolution_time_regression.csv')
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [ ]:
Experiment A — Classification
Target: escalated. Exclude ticket_id because it is an identifier rather than a meaningful predictor. Use a stratified 80/20 split so the escalation proportion is represented in both sets. The test set is reserved for one final evaluation.

In [3]:
Xc = clf_df.drop(columns=['ticket_id', 'escalated'])
yc = clf_df['escalated']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.20, random_state=42, stratify=yc)

numeric_features = Xc.select_dtypes(include=np.number).columns.tolist()
categorical_features = Xc.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
])

cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'k-NN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}
clf_rows = []
for name, model in classifiers.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    cv_out = cross_validate(pipe, Xc_train, yc_train, cv=cv_clf, scoring='f1')
    clf_rows.append({'Model': name, 'Mean CV F1': cv_out['test_score'].mean(),
                    'Std CV F1': cv_out['test_score'].std()})
clf_results = pd.DataFrame(clf_rows).sort_values('Mean CV F1', ascending=False)
display(clf_results)

,Model,Mean CV F1,Std CV F1
0,Logistic Regression,0.392651,0.062747
1,k-NN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


In [ ]:
Tune logistic regression
Tune regularization strength C on training data only. C controls inverse regularization strength: smaller values apply stronger regularization. Grid values are spaced logarithmically to test several orders of magnitude.

In [4]:
logit_pipe = Pipeline([('preprocess', preprocessor),
                       ('model', LogisticRegression(max_iter=2000, random_state=42))])
clf_search = GridSearchCV(logit_pipe, {'model__C': [0.01, 0.1, 1, 10, 100]},
                         cv=cv_clf, scoring='f1', n_jobs=-1)
clf_search.fit(Xc_train, yc_train)
print('Best parameters:', clf_search.best_params_)
print('Best CV F1:', clf_search.best_score_)

Best parameters: {'model__C': 10}
Best CV F1: 0.4225932078257659


In [ ]:
Final classification test evaluation 
After model selection, evaluate the selected pipeline on the held-out test set. Confusion-matrix layout: rows = actual class, columns = predicted class.

In [5]:
best_clf = clf_search.best_estimator_
yc_pred = best_clf.predict(Xc_test)
print('Test F1:', f1_score(yc_test, yc_pred))
print(classification_report(yc_test, yc_pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(yc_test, yc_pred))

Test F1: 0.3137254901960784
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

Confusion matrix:
 [[137   5]
 [ 30   8]]


In [ ]:
Experiment B — Regression
Target: resolution_time_hours. Exclude ticket_id. Use an 80/20 train/test split. A shuffled 5-fold KFold CV compares the three regressors using MAE. Negative MAE is the scikit-learn scoring convention, so multiply scores by -1 to report MAE.

In [6]:
Xr = reg_df.drop(columns=['ticket_id', 'resolution_time_hours'])
yr = reg_df['resolution_time_hours']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.20, random_state=42)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)
regressors = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42)
}
reg_rows = []
for name, model in regressors.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    cv_out = cross_validate(pipe, Xr_train, yr_train, cv=cv_reg, scoring='neg_mean_absolute_error')
    mae_scores = -cv_out['test_score']
    reg_rows.append({'Model': name, 'Mean CV MAE': mae_scores.mean(),
                    'Std CV MAE': mae_scores.std()})
reg_results = pd.DataFrame(reg_rows).sort_values('Mean CV MAE')
display(reg_results)

,Model,Mean CV MAE,Std CV MAE
1,Ridge Regression,2.035127,0.171404
0,Linear Regression,2.035134,0.171650
2,Decision Tree Regressor,3.239097,0.053807


In [ ]:
Tune Ridge regression
Tune alpha, the Ridge penalty strength, on training data only. Higher values penalize coefficient magnitude more strongly.

In [7]:
ridge_pipe = Pipeline([('preprocess', preprocessor), ('model', Ridge())])
ridge_search = GridSearchCV(ridge_pipe, {'model__alpha': [0.01, 0.1, 1, 10, 100]},
                           cv=cv_reg, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_search.fit(Xr_train, yr_train)
print('Best parameters:', ridge_search.best_params_)
print('Best CV MAE:', -ridge_search.best_score_)

Best parameters: {'model__alpha': 0.1}
Best CV MAE: 2.035118085555478


In [ ]:
Final regression test evaluation

In [8]:
best_ridge = ridge_search.best_estimator_
yr_pred = best_ridge.predict(Xr_test)
print('Test MAE:', mean_absolute_error(yr_test, yr_pred))
print('Test RMSE:', root_mean_squared_error(yr_test, yr_pred))

Test MAE: 2.230086144180656
Test RMSE: 3.4027561084563143


In [ ]:
Report / reflection
- Problem framing: classification predicts whether a ticket escalates; regression estimates resolution hours.
- Preprocessing: numeric missing values use median imputation then scaling; categorical missing values use most-frequent imputation and one-hot encoding. Median is less sensitive to extreme values than mean; scaling supports distance-based models and coefficient-based models; one-hot encoding turns categories into numeric indicators.
- Validation: classification uses stratified splitting and StratifiedKFold to preserve the class ratio. Regression uses shuffled 5-fold KFold. CV uses training data only; test data is reserved for final evaluation.
- Leakage prevention: all imputation, scaling, and encoding are inside the Pipeline/ColumnTransformer, so each CV fold learns preprocessing only from its training fold. The identifier is removed.
- Fold variability: standard deviation measures how sensitive performance is to different fold samples. Large variability suggests instability or data sensitivity; small variability suggests more consistent performance.
- Why stratify? Escalations are the minority class; stratification keeps roughly similar proportions in split subsets and folds, making evaluation more representative.
- Why test once? Repeated test feedback indirectly tunes decisions to the test set, turning it into another validation set and making the final estimate optimistic.
- False positives / false negatives: false positives are non-escalated tickets flagged as escalated and can waste specialist attention. False negatives are escalated tickets missed by the model and can delay intervention. Consider recall, precision, class weighting, and threshold choice depending on the operational cost.
- Outliers and MAE vs RMSE: MAE penalizes errors linearly and is more robust to extreme residuals. RMSE squares errors, so large misses carry more weight; therefore RMSE may favor a model that avoids large errors even if its typical absolute error is higher. Ticket resolution times can be right-skewed and contain unusually long cases.
- Deployment constraints: inference latency, interpretability, available features at prediction time, drift, class imbalance, escalation capacity, and the relative cost of missed escalations versus unnecessary escalations may change model choice.
- Limitations: results depend on this dataset and split; CV estimates can vary; features may omit important context; ticket patterns can change over time; performance metrics alone do not establish fairness or operational usefulness.